# Spike — thử nghiệm cơ chế SHAP theo nhóm nguyên nhân rủi ro (chuẩn bị AC cho Story #11)

Chưa phải Task chính thức — đây là thử nghiệm để có **số liệu thật** trước khi chốt cơ chế tính "nhóm nguyên nhân rủi ro chính" mà `docs/wireframes.md` đã giả định hiển thị, nhưng cơ chế cụ thể để ngỏ.

**Cơ chế ban đầu đề xuất**: SHAP values tính riêng cho từng đơn hàng (`TreeExplainer` trên XGBoost đã đóng gói ở `models/xgboost_final.pkl`), gộp `|SHAP|` theo 2 nhóm feature định trước — 🔵 Vận chuyển / 🟢 Chuẩn bị hàng, loại `order_purchase_month` (confound mùa vụ) khỏi cả 2.

**Qua thử nghiệm số liệu thật, đã điều chỉnh 3 điểm trước khi chốt làm AC chính thức của Story #11:**
1. Sửa lỗi phương pháp (mục 7-8): chỉ cộng SHAP của cột one-hot state đang bật (=1), không cộng cả 50 cột — tránh phình nhóm giả tạo (Vận chuyển từ 84% xuống 66%).
2. Đổi tên nhóm "Chuẩn bị hàng" → **"Chuẩn bị & thanh toán"** (mục 11) vì `payment_*` chiếm ~30% trọng số nhóm, không chỉ độ phức tạp đơn.
3. Nâng `order_purchase_month` thành **nhóm thứ 3 riêng "Yếu tố thời điểm"** (mục 8) thay vì loại hẳn — vì nó là driver mạnh nhất ở 60% đơn dự đoán trễ, ẩn đi sẽ gây hiểu lầm cho đúng nhóm đơn người dùng quan tâm nhất.

**Kết quả cuối cùng (3 nhóm)**: lưu ở `models/risk_group_shap_spike_summary.json`.

## 1. Nạp model + dữ liệu test (đúng pattern dùng cho API)

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
import shap

model = joblib.load("../models/xgboost_final.pkl")
with open("../models/final_model.json", encoding="utf-8") as f:
    meta = json.load(f)

feature_columns = meta["feature_columns"]
decision_threshold = meta["decision_threshold"]

test_df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)
test_df["is_delayed"] = test_df["is_delayed"].astype(bool)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    test_df[col] = test_df[col].astype("boolean")

X_test = test_df[feature_columns]
y_test = test_df["is_delayed"].astype(int)

print(f"X_test: {X_test.shape}, decision_threshold={decision_threshold:.4f}")

X_test: (19289, 76), decision_threshold=0.5391


## 2. Định nghĩa các nhóm feature, kiểm tra bao phủ đúng 76 cột

`order_purchase_month` giữ riêng (không gộp vào 2 nhóm Vận chuyển/Chuẩn bị) — ban đầu định loại hẳn khỏi tỷ lệ hiển thị, nhưng sau khi thấy nó là driver mạnh nhất ở 60% đơn dự đoán trễ (mục 8b), quyết định nâng thành **nhóm thứ 3 riêng** ("Yếu tố thời điểm") thay vì ẩn đi — xem mục 8.

In [2]:
EXCLUDED_FEATURES = {"order_purchase_month"}

SHIPPING_FEATURES = {
    col for col in feature_columns
    if col in ("seller_customer_distance_km", "estimated_delivery_days")
    or col.startswith("customer_state_")
    or col.startswith("primary_seller_state_")
}

PREP_FEATURES = set(feature_columns) - SHIPPING_FEATURES - EXCLUDED_FEATURES

# Sanity check: 3 nhom phai phu het, khong chong lan, khong thieu cot nao
assert SHIPPING_FEATURES | PREP_FEATURES | EXCLUDED_FEATURES == set(feature_columns)
assert not (SHIPPING_FEATURES & PREP_FEATURES)

print(f"Van chuyen: {len(SHIPPING_FEATURES)} feature")
print(f"Chuan bi hang: {len(PREP_FEATURES)} feature")
print(f"Loai khoi ca 2 nhom: {EXCLUDED_FEATURES}")

Van chuyen: 52 feature
Chuan bi hang: 23 feature
Loai khoi ca 2 nhom: {'order_purchase_month'}


## 3. Tính SHAP values trên toàn bộ tập test (TreeExplainer — đủ nhanh cho real-time API sau này)

In [3]:
import time

explainer = shap.TreeExplainer(model)

start = time.time()
shap_values = explainer.shap_values(X_test)
elapsed = time.time() - start

shap_df = pd.DataFrame(shap_values, columns=feature_columns, index=X_test.index)

print(f"SHAP values shape: {shap_df.shape}")
print(f"Thoi gian tinh cho {len(X_test)} don: {elapsed:.2f}s ({elapsed / len(X_test) * 1000:.3f}ms/don)")

SHAP values shape: (19289, 76)
Thoi gian tinh cho 19289 don: 0.87s (0.045ms/don)


## 4. Gộp `|SHAP|` theo 2 nhóm cho từng đơn, tính tỷ lệ %

In [4]:
shap_abs = shap_df.abs()

shipping_magnitude = shap_abs[list(SHIPPING_FEATURES)].sum(axis=1)
prep_magnitude = shap_abs[list(PREP_FEATURES)].sum(axis=1)
total_magnitude = shipping_magnitude + prep_magnitude

result = pd.DataFrame({
    "order_id": test_df["order_id"],
    "is_delayed": y_test,
    "predicted_proba": model.predict_proba(X_test)[:, 1],
    "shipping_pct": shipping_magnitude / total_magnitude * 100,
    "prep_pct": prep_magnitude / total_magnitude * 100,
})
result["predicted_delayed"] = (result["predicted_proba"] >= decision_threshold).astype(int)
result["dominant_group"] = np.where(result["shipping_pct"] >= 50, "Van chuyen", "Chuan bi hang")

result.describe()

,is_delayed,predicted_proba,shipping_pct,prep_pct,predicted_delayed
count,19289.000000,19289.000000,19289.000000,19289.000000,19289.000000
mean,0.081134,0.236945,62.054279,37.945724,0.113329
std,0.273048,0.206611,12.224097,12.224097,0.317003
min,0.000000,0.000207,11.071671,5.304825,0.000000
25%,0.000000,0.081986,54.475903,29.046762,0.000000
50%,0.000000,0.168822,63.131599,36.868397,0.000000
75%,0.000000,0.331489,70.953247,45.524090,0.000000
max,1.000000,0.962189,94.695168,88.928329,1.000000


## 5. Kiểm tra phân phối — có degenerate (luôn lệch 1 nhóm) không? Có khác nhau giữa các đơn không?

In [5]:
print("Phan phoi dominant_group tren toan bo test set:")
print(result["dominant_group"].value_counts(normalize=True).mul(100).round(2))

print("\nPhan phoi dominant_group CHI tren cac don model du doan TRE (nhom se hien thi tren dashboard):")
delayed_pred = result[result["predicted_delayed"] == 1]
print(f"So don du doan tre: {len(delayed_pred)}")
print(delayed_pred["dominant_group"].value_counts(normalize=True).mul(100).round(2))

print("\nPhan vi cua shipping_pct (toan bo test set):")
print(result["shipping_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

extreme_pct = ((result["shipping_pct"] >= 95) | (result["shipping_pct"] <= 5)).mean() * 100
print(f"\n% don co ty le lech cuc doan (>=95% hoac <=5% ve 1 nhom): {extreme_pct:.2f}%")

print("\n10 don du doan tre ngau nhien, xem ty le co da dang khong:")
print(delayed_pred[["order_id", "predicted_proba", "shipping_pct", "prep_pct", "dominant_group"]].sample(10, random_state=42))

Phan phoi dominant_group tren toan bo test set:
dominant_group
Van chuyen       84.08
Chuan bi hang    15.92
Name: proportion, dtype: float64

Phan phoi dominant_group CHI tren cac don model du doan TRE (nhom se hien thi tren dashboard):
So don du doan tre: 2186
dominant_group
Van chuyen       90.35
Chuan bi hang     9.65
Name: proportion, dtype: float64

Phan vi cua shipping_pct (toan bo test set):
count    19289.000000
mean        62.054279
std         12.224097
min         11.071671
5%          39.972383
25%         54.475903
50%         63.131599
75%         70.953247
95%         79.984982
max         94.695168
Name: shipping_pct, dtype: float64

% don co ty le lech cuc doan (>=95% hoac <=5% ve 1 nhom): 0.00%

10 don du doan tre ngau nhien, xem ty le co da dang khong:
                               order_id  predicted_proba  shipping_pct  \
6800   b4d298d0232cfc779cbfdfcf795e7934         0.598015     81.674965   
11692  d4a3bd4ef8590f7923183b70e32326d8         0.665294     51.82672

## 7. Kiểm tra nghi vấn: tỷ lệ lệch về "Vận chuyển" có phải do CỘNG DỒN 50 cột one-hot state (49 trong đó =0 cho mỗi đơn) không, hay là tín hiệu thật từ 2 cột số cốt lõi (`seller_customer_distance_km`, `estimated_delivery_days`)?

In [6]:
state_dummy_cols = [c for c in SHIPPING_FEATURES if c.startswith("customer_state_") or c.startswith("primary_seller_state_")]
core_shipping_cols = ["seller_customer_distance_km", "estimated_delivery_days"]

# |SHAP| trung binh cong don theo tung nguon, tren toan bo test set
state_dummies_magnitude = shap_abs[state_dummy_cols].sum(axis=1)
core_magnitude = shap_abs[core_shipping_cols].sum(axis=1)

print("Dong gop trung binh vao tong 'Van chuyen' magnitude:")
print(f"  50 cot one-hot state (49/50 =0 cho moi don): {state_dummies_magnitude.mean():.4f} "
      f"({state_dummies_magnitude.mean() / shipping_magnitude.mean() * 100:.1f}% cua nhom)")
print(f"  2 cot so cot loi (distance + estimated_delivery_days): {core_magnitude.mean():.4f} "
      f"({core_magnitude.mean() / shipping_magnitude.mean() * 100:.1f}% cua nhom)")

# So sanh: neu CHI dung 2 cot so cot loi (khong cong don one-hot) thi ty le nhom se the nao?
result["shipping_pct_core_only"] = core_magnitude / (core_magnitude + prep_magnitude) * 100
print("\nNeu nhom 'Van chuyen' CHI gom 2 cot so cot loi (bo one-hot state):")
print(result["shipping_pct_core_only"].describe(percentiles=[0.25, 0.5, 0.75]))
print("\ndominant_group neu dung shipping_pct_core_only:")
print((result["shipping_pct_core_only"] >= 50).value_counts(normalize=True).mul(100).round(2))

Dong gop trung binh vao tong 'Van chuyen' magnitude:
  50 cot one-hot state (49/50 =0 cho moi don): 0.8187 (46.6% cua nhom)
  2 cot so cot loi (distance + estimated_delivery_days): 0.9365 (53.4% cua nhom)

Neu nhom 'Van chuyen' CHI gom 2 cot so cot loi (bo one-hot state):
count    19289.000000
mean        44.418385
std         16.960148
min          0.557083
25%         31.944948
50%         45.054447
75%         57.070763
max         89.430595
Name: shipping_pct_core_only, dtype: float64

dominant_group neu dung shipping_pct_core_only:
shipping_pct_core_only
False    60.56
True     39.44
Name: proportion, dtype: float64


## 8. Sửa lại cơ chế: chỉ cộng SHAP của cột one-hot state ĐANG BẬT (=1) cho từng đơn, không cộng cả 50 cột

Cộng dồn `|SHAP|` qua toàn bộ cột one-hot (49/50 luôn =0) là sai phương pháp — làm phình nhóm có nhiều cột one-hot hơn một cách giả tạo, không phản ánh tín hiệu thật. Cách đúng: nhân `|SHAP|` với chính giá trị one-hot (0/1) trước khi cộng — chỉ giữ lại đóng góp của đúng 1 state đang áp dụng cho đơn đó.

In [7]:
# X_test[state_dummy_cols] la 0/1 -> nhan voi shap_abs se tu dong loai bo cot dang tat (=0)
active_state_magnitude = (shap_abs[state_dummy_cols] * X_test[state_dummy_cols]).sum(axis=1)
shipping_magnitude_fixed = active_state_magnitude + core_magnitude
seasonal_magnitude = shap_abs["order_purchase_month"]

PREP_GROUP_LABEL = "Chuan bi & thanh toan"  # doi ten tu "Chuan bi hang" vi gom ca payment_* (~30% trong so nhom)
SEASONAL_GROUP_LABEL = "Yeu to thoi diem"  # nhom thu 3 rieng cho order_purchase_month (khong con loai bo)

total_magnitude_3way = shipping_magnitude_fixed + prep_magnitude + seasonal_magnitude

result["shipping_pct_fixed"] = shipping_magnitude_fixed / total_magnitude_3way * 100
result["prep_pct_fixed"] = prep_magnitude / total_magnitude_3way * 100
result["seasonal_pct"] = seasonal_magnitude / total_magnitude_3way * 100

group_pcts = result[["shipping_pct_fixed", "prep_pct_fixed", "seasonal_pct"]]
group_pcts.columns = ["Van chuyen", PREP_GROUP_LABEL, SEASONAL_GROUP_LABEL]
result["dominant_group_fixed"] = group_pcts.idxmax(axis=1)

print("Phan phoi dominant_group_fixed (3 nhom) tren toan bo test set:")
print(result["dominant_group_fixed"].value_counts(normalize=True).mul(100).round(2))

delayed_pred_fixed = result[result["predicted_delayed"] == 1]
print("\nPhan phoi dominant_group_fixed CHI tren don du doan TRE:")
print(delayed_pred_fixed["dominant_group_fixed"].value_counts(normalize=True).mul(100).round(2))

print("\nPhan vi shipping_pct_fixed (toan bo test set):")
print(result["shipping_pct_fixed"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

extreme_pct_fixed = ((result["shipping_pct_fixed"] >= 95) | (result["shipping_pct_fixed"] <= 5)).mean() * 100
print(f"\n% don co ty le lech cuc doan cho Van chuyen (>=95% hoac <=5%): {extreme_pct_fixed:.2f}%")

Phan phoi dominant_group_fixed (3 nhom) tren toan bo test set:
dominant_group_fixed
Van chuyen               60.32
Chuan bi & thanh toan    28.54
Yeu to thoi diem         11.13
Name: proportion, dtype: float64

Phan phoi dominant_group_fixed CHI tren don du doan TRE:
dominant_group_fixed
Van chuyen               59.47
Chuan bi & thanh toan    23.83
Yeu to thoi diem         16.70
Name: proportion, dtype: float64

Phan vi shipping_pct_fixed (toan bo test set):
count    19289.000000
mean        44.143799
std         13.848148
min          5.496276
5%          22.260857
25%         34.082306
50%         43.611767
75%         53.962845
95%         67.516690
max         89.629356
Name: shipping_pct_fixed, dtype: float64

% don co ty le lech cuc doan cho Van chuyen (>=95% hoac <=5%): 0.00%


## 9. Lưu tóm tắt kết quả CUỐI CÙNG (cơ chế đã sửa — khuyến nghị cho AC Story #11)

In [8]:
final_summary = {
    "mechanism": (
        "SHAP TreeExplainer per-instance, 3 nhom: 'Van chuyen' = |SHAP| cua "
        "seller_customer_distance_km + estimated_delivery_days + CHI cot one-hot "
        "customer_state/primary_seller_state DANG BAT (=1) cho don do (khong cong het 50 cot). "
        f"'{PREP_GROUP_LABEL}' = tong |SHAP| cua 23 cot (doi ten tu 'Chuan bi hang' vi gom ca payment_*, "
        f"~30% trong so nhom). '{SEASONAL_GROUP_LABEL}' = |SHAP| cua order_purchase_month rieng "
        "(ban dau dinh loai hoan toan vi la confound mua vu, nhung la driver manh nhat o 60% don du doan tre "
        "nen nang thanh nhom thu 3 rieng thay vi an di)."
    ),
    "rejected_naive_approach": (
        "Cong don |SHAP| qua toan bo 50 cot one-hot state (ke ca 49 cot =0) lam nhom Van chuyen "
        "bi phinh gia tao: 46.6% magnitude cua nhom den tu cac cot dang TAT. "
        "Ket qua sai lech: Van chuyen chiem 84% (90% neu chi xet don du doan tre) mot cach khong cong bang."
    ),
    "avg_shap_compute_time_ms_per_order": elapsed / len(X_test) * 1000,
    "dominant_group_pct_all_test": result["dominant_group_fixed"].value_counts(normalize=True).mul(100).round(2).to_dict(),
    "dominant_group_pct_predicted_delayed_only": delayed_pred_fixed["dominant_group_fixed"].value_counts(normalize=True).mul(100).round(2).to_dict(),
    "shipping_pct_percentiles_all_test": result["shipping_pct_fixed"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_dict(),
    "seasonal_pct_percentiles_all_test": result["seasonal_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_dict(),
    "pct_orders_extreme_split_ge95_or_le5": round(extreme_pct_fixed, 2),
    "shipping_features": sorted(SHIPPING_FEATURES),
    "prep_features": sorted(PREP_FEATURES),
    "seasonal_features": sorted(EXCLUDED_FEATURES),
}

with open("../models/risk_group_shap_spike_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)

print("Da luu models/risk_group_shap_spike_summary.json (co che 3 nhom)")
print(json.dumps(final_summary, indent=2, ensure_ascii=False))

Da luu models/risk_group_shap_spike_summary.json (co che 3 nhom)
{
  "mechanism": "SHAP TreeExplainer per-instance, 3 nhom: 'Van chuyen' = |SHAP| cua seller_customer_distance_km + estimated_delivery_days + CHI cot one-hot customer_state/primary_seller_state DANG BAT (=1) cho don do (khong cong het 50 cot). 'Chuan bi & thanh toan' = tong |SHAP| cua 23 cot (doi ten tu 'Chuan bi hang' vi gom ca payment_*, ~30% trong so nhom). 'Yeu to thoi diem' = |SHAP| cua order_purchase_month rieng (ban dau dinh loai hoan toan vi la confound mua vu, nhung la driver manh nhat o 60% don du doan tre nen nang thanh nhom thu 3 rieng thay vi an di).",
  "rejected_naive_approach": "Cong don |SHAP| qua toan bo 50 cot one-hot state (ke ca 49 cot =0) lam nhom Van chuyen bi phinh gia tao: 46.6% magnitude cua nhom den tu cac cot dang TAT. Ket qua sai lech: Van chuyen chiem 84% (90% neu chi xet don du doan tre) mot cach khong cong bang.",
  "avg_shap_compute_time_ms_per_order": 0.04530571019880026,
  "dominant_group

## 10. Ví dụ chi tiết trên vài đơn thật — xem breakdown từng feature, không chỉ % tổng

Chọn 5 đơn dự đoán trễ (xác suất cao/thấp khác nhau), in ra top 6 feature đóng góp SHAP lớn nhất kèm dấu (+ đẩy xác suất trễ lên, - kéo xuống) và nhóm được gán, để đánh giá trực giác có hợp lý không.

In [9]:
def group_of(feature):
    if feature in PREP_FEATURES:
        return PREP_GROUP_LABEL
    if feature in EXCLUDED_FEATURES:
        return SEASONAL_GROUP_LABEL
    return "Van chuyen"

sample_positions = delayed_pred_fixed.sort_values("predicted_proba", ascending=False).iloc[[0, 1]].index.tolist()
sample_positions += delayed_pred_fixed.iloc[(delayed_pred_fixed["predicted_proba"] - 0.55).abs().argsort()[:2]].index.tolist()
sample_positions += delayed_pred_fixed.sort_values("predicted_proba").iloc[[0]].index.tolist()

for pos in sample_positions:
    row = result.loc[pos]
    print(f"\n=== order_id={row['order_id']}  proba={row['predicted_proba']:.3f}  "
          f"shipping={row['shipping_pct_fixed']:.1f}%  prep={row['prep_pct_fixed']:.1f}%  "
          f"seasonal={row['seasonal_pct']:.1f}%  dominant={row['dominant_group_fixed']} ===")
    row_shap = shap_df.loc[pos]
    # Chi xet feature dang "active": one-hot phai =1, con lai xet gia tri thuc
    active_mask = pd.Series(True, index=feature_columns)
    for c in state_dummy_cols:
        active_mask[c] = X_test.loc[pos, c] == 1
    top_features = row_shap[active_mask].abs().sort_values(ascending=False).head(6).index
    for feat in top_features:
        print(f"  {feat:35s} shap={row_shap[feat]:+.4f}  value={X_test.loc[pos, feat]!s:>10s}  nhom={group_of(feat)}")


=== order_id=9c57e2e7098196c6c6ac9d333b0395b0  proba=0.962  shipping=61.1%  prep=35.7%  seasonal=3.2%  dominant=Van chuyen ===
  estimated_delivery_days             shap=+3.4565  value=4.384247685185185  nhom=Van chuyen
  approval_gap_hours                  shap=+0.8548  value=46.07277777777778  nhom=Chuan bi & thanh toan
  items_total_weight_g                shap=-0.3198  value=     200.0  nhom=Chuan bi & thanh toan
  items_total_price                   shap=-0.2882  value=      16.9  nhom=Chuan bi & thanh toan
  items_total_freight                 shap=-0.2810  value=      7.39  nhom=Chuan bi & thanh toan
  payment_total_value                 shap=+0.2122  value=     24.29  nhom=Chuan bi & thanh toan

=== order_id=128e2bcc764e1cf5e4952af2267acef2  proba=0.962  shipping=71.2%  prep=25.7%  seasonal=3.1%  dominant=Van chuyen ===
  estimated_delivery_days             shap=+3.5353  value=3.157627314814815  nhom=Van chuyen
  approval_gap_hours                  shap=+0.7671  value=33.51861

## 11. Đánh giá lại gán nhóm: trong "Chuẩn bị hàng", `payment_*` (thanh toán) đóng góp bao nhiêu so với `items_*` (độ phức tạp đơn) và `approval_gap_hours`?

`payment_*` mô tả cách khách trả tiền, không hẳn là "seller đóng gói chậm" theo nghĩa đen — xếp vào đây chỉ vì không thuộc "Vận chuyển". Kiểm tra tỷ trọng thật để biết đây có phải mối lo đáng kể hay không.

In [10]:
payment_cols = [c for c in PREP_FEATURES if c.startswith("payment_")]
items_cols = [c for c in PREP_FEATURES if c.startswith("items_")]
approval_cols = [c for c in PREP_FEATURES if c == "approval_gap_hours"]

payment_mag = shap_abs[payment_cols].sum(axis=1)
items_mag = shap_abs[items_cols].sum(axis=1)
approval_mag = shap_abs[approval_cols].sum(axis=1)

print("Ty trong trung binh trong nhom 'Chuan bi hang' (tren toan bo test set):")
print(f"  items_* ({len(items_cols)} cot, do phuc tap don):      {items_mag.mean() / prep_magnitude.mean() * 100:.1f}%")
print(f"  payment_* ({len(payment_cols)} cot, cach thanh toan):    {payment_mag.mean() / prep_magnitude.mean() * 100:.1f}%")
print(f"  approval_gap_hours (1 cot, thoi gian duyet don):  {approval_mag.mean() / prep_magnitude.mean() * 100:.1f}%")

Ty trong trung binh trong nhom 'Chuan bi hang' (tren toan bo test set):
  items_* (8 cot, do phuc tap don):      53.4%
  payment_* (14 cot, cach thanh toan):    29.5%
  approval_gap_hours (1 cot, thoi gian duyet don):  17.1%
